# Week 01B — Colour and Histograms

**MCTA 4364 Machine Vision** · Session 2 of Week 1 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W01_images_colour/W01B_colour_histograms.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W01_images_colour/W01B_colour_histograms.ipynb)

Colour is one of the cheapest and most powerful cues in machine vision: sorting fruit, checking wire colours, reading traffic
lights, detecting rust. But "colour" in a camera is just three numbers per pixel that change with the **light source**, the
**sensor** and **shading**. This session shows how cameras sense colour, which **colour space** makes a task easy, and how
**histograms** summarise an image.

### Learning outcomes
By the end of this session you will be able to:
1. Explain how a single-sensor camera captures colour with a **Bayer filter** and **demosaicing**.
2. Convert between **BGR/RGB, HSV, Lab and YCrCb**, and choose the right space for a task.
3. Segment and count objects by colour with `cv2.inRange`, including the **red hue wrap-around**.
4. Compute, interpret and compare **histograms**; use **back-projection** to find a colour model in an image.
5. Correct a colour cast with **grey-world white balance**.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–12 | 1–2 | Light, sensors and the Bayer filter |
| 12–35 | 3–5 | Colour spaces: RGB cube, HSV, Lab (interactive 3D) |
| 35–50 | 6 | Colour segmentation and counting (interactive HSV tuner) |
| 50–65 | 7–8 | Histograms, comparison and back-projection |
| 65–72 | 9 | White balance |
| 72–80 | 12 | Quiz and exit ticket |
| Home | 10–11 | Exercises with self-checks, colour-sorting challenge |

**Prerequisites:** Week 01A (images as arrays, BGR vs RGB).

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from matplotlib.colors import hsv_to_rgb

from cvhelpers import check, interact, load_sample, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
smarties = load_sample("smarties.png")     # OpenCV sample images (BGR)
fruits = load_sample("fruits.jpg")
apple, orange = load_sample("apple.jpg"), load_sample("orange.jpg")
show(smarties, fruits, apple, orange, titles=["smarties.png", "fruits.jpg", "apple.jpg", "orange.jpg"], figsize=(20, 4.2))

## 1. Light and colour

Visible light is electromagnetic radiation with wavelengths of roughly **380–750 nm**. Machine vision also uses the neighbouring bands:
**ultraviolet** (fluorescence inspection) and **near-infrared (NIR, 750–1000 nm)**, which silicon sensors still detect. NIR sees through
some plastics and dyes and is insensitive to visible colour changes.

A camera pixel's value in channel $k$ is (roughly) the light spectrum reaching it weighted by the channel's sensitivity:

$$ I_k = \int L(\lambda)\, R(\lambda)\, S_k(\lambda)\, d\lambda $$

where $L$ is the **illuminant**, $R$ the **surface reflectance** and $S_k$ the **sensor sensitivity**. Change the lamp ($L$) and every
pixel value changes, even though the object ($R$) did not. That is why lighting control and white balance (Section 9) matter.

In [ ]:
def wavelength_to_rgb(nm):
    """Approximate display colour of a visible wavelength (Bruton's piecewise approximation)."""
    r = g = b = 0.0
    if 380 <= nm < 440: r, b = -(nm - 440) / 60, 1.0
    elif 440 <= nm < 490: g, b = (nm - 440) / 50, 1.0
    elif 490 <= nm < 510: g, b = 1.0, -(nm - 510) / 20
    elif 510 <= nm < 580: r, g = (nm - 510) / 70, 1.0
    elif 580 <= nm < 645: r, g = 1.0, -(nm - 645) / 65
    elif 645 <= nm <= 750: r = 1.0
    fade = 1.0 if 420 <= nm <= 700 else 0.35
    return np.array([r, g, b]) * fade


nm = np.arange(300, 1001)
strip = np.array([wavelength_to_rgb(w) if 380 <= w <= 750 else ([0.55, 0.3, 0.75] if w < 380 else [0.45, 0.1, 0.1])
                  for w in nm])
fig, axes = plt.subplots(2, 1, figsize=(15, 5), gridspec_kw={"height_ratios": [1, 2]}, sharex=True)
axes[0].imshow(strip[None], aspect="auto", extent=[300, 1000, 0, 1]); axes[0].set_yticks([])
for x, label in [(340, "UV"), (565, "visible"), (875, "near-infrared (NIR)")]:
    axes[0].text(x, 0.5, label, ha="center", va="center", color="w", fontweight="bold")
for mu, sd, color, name in [(460, 25, "#1a73e8", "blue channel"), (540, 35, "#188038", "green channel"), (600, 30, "#d93025", "red channel")]:
    axes[1].plot(nm, np.exp(-0.5 * ((nm - mu) / sd) ** 2), color=color, lw=2.5, label=name)
qe = np.clip(np.exp(-0.5 * ((nm - 620) / 190) ** 2), 0, 1)
axes[1].plot(nm, qe, "k--", lw=1.5, label="bare silicon sensitivity")
axes[1].axvspan(700, 1000, color="#fad2cf", alpha=0.3); axes[1].text(850, 0.9, "IR-cut filter blocks this\nin colour cameras", ha="center")
axes[1].set_xlabel("wavelength (nm)"); axes[1].set_ylabel("relative sensitivity"); axes[1].legend(loc="upper left")
plt.suptitle("Figure 1.1 - The spectrum and (illustrative) camera channel sensitivities", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. How a camera sees colour: the Bayer filter

Each pixel of a silicon sensor only measures *how much* light arrives, not its colour. Most colour cameras put a mosaic of red,
green and blue filters over the pixels: the **Bayer pattern** (Bayer, 1976), with twice as many green pixels because our eyes are most
sensitive to green. Each pixel records **one** colour; the other two are **interpolated** from neighbours (**demosaicing**).
This is why colour cameras have lower effective resolution and can show false colour ("zipper") at sharp edges. For precise
measurement, machine-vision engineers often prefer **monochrome** cameras.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
pattern = np.zeros((8, 8, 3))
for r in range(8):
    for c in range(8):
        pattern[r, c] = [0, 0, 1] if (r % 2, c % 2) == (0, 0) else [1, 0, 0] if (r % 2, c % 2) == (1, 1) else [0, 0.8, 0]
axes[0].imshow(pattern)                                # RGB display colours
axes[0].set_title("Bayer pattern (BGGR): 50% G, 25% R, 25% B")
for k in range(9):
    axes[0].axhline(k - 0.5, color="w", lw=1); axes[0].axvline(k - 0.5, color="w", lw=1)

# Simulate the raw sensor: keep only the filtered colour at each pixel (BGGR layout)
patch = smarties[200:330, 50:250]
raw = np.zeros(patch.shape[:2], np.uint8)
raw[0::2, 0::2] = patch[0::2, 0::2, 0]   # B
raw[0::2, 1::2] = patch[0::2, 1::2, 1]   # G
raw[1::2, 0::2] = patch[1::2, 0::2, 1]   # G
raw[1::2, 1::2] = patch[1::2, 1::2, 2]   # R
mosaic_vis = np.zeros_like(patch)
mosaic_vis[0::2, 0::2, 0] = raw[0::2, 0::2]; mosaic_vis[0::2, 1::2, 1] = raw[0::2, 1::2]
mosaic_vis[1::2, 0::2, 1] = raw[1::2, 0::2]; mosaic_vis[1::2, 1::2, 2] = raw[1::2, 1::2]
demosaiced = cv2.cvtColor(raw, cv2.COLOR_BayerRG2BGR)     # OpenCV names the pattern from the 2nd row/col
axes[1].imshow(raw, cmap="gray"); axes[1].set_title("raw sensor data (one value per pixel)")
axes[2].imshow(cv2.cvtColor(mosaic_vis, cv2.COLOR_BGR2RGB)); axes[2].set_title("same data coloured by filter")
axes[3].imshow(cv2.cvtColor(demosaiced, cv2.COLOR_BGR2RGB)); axes[3].set_title("after demosaicing (cv2.cvtColor)")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Figure 2.1 - From a single-sensor mosaic to a full colour image", fontweight="bold")
plt.tight_layout(); plt.show()
err = np.abs(demosaiced.astype(int) - patch.astype(int)).mean()
print(f"Mean demosaicing error: {err:.1f} grey levels (largest at sharp colour edges)")

## 3. The RGB colour cube

Each pixel is a point in a 3D cube with axes R, G and B (0–255). The interactive plot shows 3,000 random pixels of `fruits.jpg`,
each drawn in its own colour. Notice that the pixels of one fruit form an **elongated cloud** pointing towards black: the same
surface appears brighter or darker depending on shading. In RGB, *colour* and *brightness* are mixed together.

In [ ]:
rng = np.random.default_rng(0)
px = fruits.reshape(-1, 3)[rng.choice(fruits.shape[0] * fruits.shape[1], 3000, replace=False)]
rgb = px[:, ::-1]
fig = go.Figure(go.Scatter3d(x=rgb[:, 0], y=rgb[:, 1], z=rgb[:, 2], mode="markers",
                             marker=dict(size=2.5, color=[f"rgb({r},{g},{b})" for r, g, b in rgb])))
fig.update_layout(title="Figure 3.1 - Pixels of fruits.jpg in the RGB cube (drag to rotate)", height=550,
                  scene=dict(xaxis_title="R", yaxis_title="G", zaxis_title="B",
                             xaxis=dict(range=[0, 255]), yaxis=dict(range=[0, 255]), zaxis=dict(range=[0, 255])))
fig.show()

## 4. HSV: separating *colour* from *brightness*

**HSV** re-arranges RGB into a cylinder:
- **Hue (H)**: *which* colour, an angle around the colour wheel. OpenCV stores 0–179 (degrees ÷ 2) so it fits in `uint8`.
- **Saturation (S)**: how *pure* the colour is (0 = grey, 255 = vivid).
- **Value (V)**: how *bright* ($V = \max(R, G, B)$).

Because shading mostly changes V, a coloured object keeps a nearly constant **hue**, which makes HSV the standard space for colour
thresholding. Two catches: hue is **circular** (red sits at both 0 and 179) and hue is **meaningless for grey pixels** (low S) and very dark pixels (low V).

In [ ]:
fig = plt.figure(figsize=(18, 5.2))
ax = fig.add_subplot(1, 3, 1, projection="polar")
theta = np.linspace(0, 2 * np.pi, 360)
for t in theta:
    ax.bar(t, 1, width=2 * np.pi / 360, bottom=0.6, color=hsv_to_rgb([t / (2 * np.pi), 1, 1]), edgecolor="none")
ax.set_yticks([])
ax.set_xticks(np.deg2rad(np.arange(0, 360, 60)),
              [f"{d} deg\nOpenCV H={d // 2}" for d in range(0, 360, 60)], fontsize=8)
ax.set_title("Hue wheel (OpenCV stores degrees / 2)")
for k, h in enumerate([0.0, 0.33, 0.66]):
    s, v = np.meshgrid(np.linspace(0, 1, 100), np.linspace(1, 0, 100))
    if k == 0:
        continue
    ax2 = fig.add_subplot(1, 3, k + 1)
    ax2.imshow(hsv_to_rgb(np.stack([np.full_like(s, h), s, v], -1)), extent=[0, 255, 0, 255])
    ax2.set_xlabel("Saturation S"); ax2.set_ylabel("Value V")
    ax2.set_title(f"slice through the cylinder at H = {int(h * 180)} (OpenCV)")
plt.suptitle("Figure 4.1 - The HSV colour model", fontweight="bold")
plt.tight_layout(); plt.show()

hsv = cv2.cvtColor(smarties, cv2.COLOR_BGR2HSV)
H, S, V = cv2.split(hsv)
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
axes[0].imshow(cv2.cvtColor(smarties, cv2.COLOR_BGR2RGB)); axes[0].set_title("input")
im = axes[1].imshow(H, cmap="hsv", vmin=0, vmax=179); axes[1].set_title("H (hue, 0-179)"); plt.colorbar(im, ax=axes[1], fraction=0.04)
axes[2].imshow(S, cmap="gray"); axes[2].set_title("S (saturation): background is ~0")
axes[3].imshow(V, cmap="gray"); axes[3].set_title("V (value = brightness)")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 4.2 - HSV channels of smarties.png", fontweight="bold")
plt.tight_layout(); plt.show()

**Figure 4.3 — Why HSV is robust to shading.** We darken the image step by step (like a shadow or an ageing lamp) and follow one red candy. Its B, G, R values collapse, but its hue barely moves.

In [ ]:
ys, xs = 245, 120      # a pixel on a red candy
gains = np.linspace(1.0, 0.25, 8)
vals = []
for gsc in gains:
    dim = np.clip(smarties.astype(np.float32) * gsc, 0, 255).astype(np.uint8)
    mean_bgr = dim[ys - 3:ys + 3, xs - 3:xs + 3].reshape(-1, 3).mean(0)
    # Convert the MEAN colour to HSV. Never average hue values directly: 178 and 2 are both red,
    # but their arithmetic mean (90) is green!
    h, s, v = cv2.cvtColor(np.uint8([[mean_bgr]]), cv2.COLOR_BGR2HSV)[0, 0]
    vals.append((mean_bgr[2], mean_bgr[1], mean_bgr[0], h, s, v))
vals = np.array(vals)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].imshow(cv2.cvtColor(np.clip(smarties * 0.4, 0, 255).astype(np.uint8), cv2.COLOR_BGR2RGB)); axes[0].plot(xs, ys, "w+", ms=18, mew=3)
axes[0].set_title("same scene at 40% light (tracked pixel +)"); axes[0].axis("off")
for k, (name, color) in enumerate([("R", "#d93025"), ("G", "#188038"), ("B", "#1a73e8")]):
    axes[1].plot(gains * 100, vals[:, k], "o-", color=color, label=name)
axes[1].set_title("RGB values change strongly"); axes[1].legend()
axes[2].plot(gains * 100, vals[:, 3], "o-", color="#e37400", label="H (hue)")
axes[2].plot(gains * 100, vals[:, 4], "s--", color="#9aa0a6", label="S")
axes[2].plot(gains * 100, vals[:, 5], "^--", color="k", label="V")
axes[2].set_title("hue stays constant; only V follows the light"); axes[2].legend()
for ax in axes[1:]:
    ax.set_xlabel("illumination (%)"); ax.invert_xaxis(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Lab and YCrCb

**CIE L\*a\*b\*** (Lab) is designed to be **perceptually uniform**: equal distances correspond to roughly equal perceived colour
differences. $L$ is lightness, $a$ runs green → red, $b$ runs blue → yellow. The colour difference between two samples is

$$ \Delta E_{ab} = \sqrt{(\Delta L)^2 + (\Delta a)^2 + (\Delta b)^2} $$

As a rule of thumb, $\Delta E < 1$ is invisible and $\Delta E > 5$ is a clearly different colour. Paint, textile and food industries
specify colour tolerances in $\Delta E$ (modern standards use the refined CIEDE2000 formula).

**YCrCb** separates luma (Y) from two colour-difference channels; it is used in JPEG and video and is popular for skin detection.

In [ ]:
lab = cv2.cvtColor(smarties, cv2.COLOR_BGR2LAB)
ycc = cv2.cvtColor(smarties, cv2.COLOR_BGR2YCrCb)
fig, axes = plt.subplots(1, 4, figsize=(20, 4.8))
L, A, B = cv2.split(lab)
axes[0].imshow(L, cmap="gray"); axes[0].set_title("L* (lightness)")
axes[1].imshow(A, cmap="RdYlGn_r"); axes[1].set_title("a* (green <-> red)")
axes[2].imshow(B, cmap="coolwarm"); axes[2].set_title("b* (blue <-> yellow)")
sel = rng.choice(L.size, 4000, replace=False)
colors = cv2.cvtColor(smarties, cv2.COLOR_BGR2RGB).reshape(-1, 3)[sel] / 255
axes[3].scatter(A.ravel()[sel].astype(int) - 128, B.ravel()[sel].astype(int) - 128, c=colors, s=4)
axes[3].axhline(0, color="k", lw=0.5); axes[3].axvline(0, color="k", lw=0.5)
axes[3].set_xlabel("a*"); axes[3].set_ylabel("b*"); axes[3].set_title("pixels in the a*b* plane: clusters = colours")
for ax in axes[:3]:
    ax.axis("off")
plt.suptitle("Figure 5.1 - Lab separates lightness from two colour-opponent axes", fontweight="bold")
plt.tight_layout(); plt.show()


def delta_e(bgr1, bgr2):
    """CIE76 colour difference between two BGR colours (OpenCV 8-bit Lab is rescaled to true Lab)."""
    lab = cv2.cvtColor(np.uint8([[bgr1, bgr2]]), cv2.COLOR_BGR2LAB).astype(float)[0]
    lab[:, 0] *= 100 / 255; lab[:, 1:] -= 128
    return float(np.linalg.norm(lab[0] - lab[1]))


pairs = {"red vs slightly darker red": ([40, 40, 200], [36, 36, 185]), "red vs orange": ([40, 40, 200], [30, 110, 230]),
         "grey 128 vs grey 131": ([128] * 3, [131] * 3), "blue vs green": ([200, 60, 30], [60, 180, 40])}
for name, (c1, c2) in pairs.items():
    print(f"{name:28s} dE = {delta_e(c1, c2):5.1f}")

## 6. Colour segmentation and counting

`cv2.inRange(hsv, lower, upper)` returns a **binary mask**: 255 where all three channels lie inside the bounds, 0 elsewhere.
We require a minimum saturation and value so that the white background and dark shadows are excluded. **Red** needs two ranges
because its hue wraps around from 179 back to 0. A small morphological opening removes noise pixels (Week 3), and `connectedComponentsWithStats` finds the separate
blobs (Week 4 covers this in depth).

In [ ]:
COLOUR_RANGES = {            # (H low, H high) in OpenCV units; red wraps around
    "red": [(0, 3), (165, 179)], "orange": [(4, 20)], "yellow": [(21, 35)],
    "green": [(36, 85)], "blue": [(90, 130)], "purple/pink": [(131, 169)]}
MIN_S, MIN_V = 90, 100          # MIN_V excludes the dark brown candies and shadows


def colour_mask(hsv_img, ranges, min_s=MIN_S, min_v=MIN_V):
    mask = np.zeros(hsv_img.shape[:2], np.uint8)
    for lo, hi in ranges:
        mask |= cv2.inRange(hsv_img, np.array([lo, min_s, min_v]), np.array([hi, 255, 255]))
    return cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))


masks = {name: colour_mask(hsv, ranges) for name, ranges in COLOUR_RANGES.items()}
fig, axes = plt.subplots(1, len(masks), figsize=(21, 3.6))
for ax, (name, m) in zip(axes, masks.items()):
    ax.imshow(cv2.cvtColor(cv2.bitwise_and(smarties, smarties, mask=m), cv2.COLOR_BGR2RGB))
    ax.set_title(name); ax.axis("off")
plt.suptitle("Figure 6.1 - Naive approach: one per-pixel HSV mask per colour. Highlights and edges split candies between classes",
             fontweight="bold")
plt.tight_layout(); plt.show()

**A robust sorter.** Per-pixel decisions are noisy: a red candy's highlight has a slightly different hue from its body. Machine-vision
systems therefore (1) segment *all* coloured objects first (high saturation), (2) measure each blob's **average colour**, and
(3) classify the whole blob. Hue is an angle, so its average must be a **circular mean**:
$\bar{h} = \operatorname{atan2}\big(\sum \sin h_i, \sum \cos h_i\big)$. (The arithmetic mean of 178 and 2 would be 90: green!)

In [ ]:
objects = cv2.morphologyEx(cv2.inRange(hsv, np.array([0, MIN_S, MIN_V]), np.array([179, 255, 255])),
                           cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
n, labels, stats, centroids = cv2.connectedComponentsWithStats(objects)


def circular_mean_hue(h_values):
    ang = h_values.astype(float) * np.pi / 90          # OpenCV hue 0-179 -> radians
    return (np.degrees(np.arctan2(np.sin(ang).mean(), np.cos(ang).mean())) % 360) / 2


def classify_hue(h):
    for name, ranges in COLOUR_RANGES.items():
        if any(lo <= h < hi + 1 for lo, hi in ranges):   # half-open: a mean hue can be 179.4
            return name
    return "unknown"


counts = {name: 0 for name in COLOUR_RANGES}
vis = cv2.cvtColor(smarties, cv2.COLOR_BGR2RGB).copy()
for i in range(1, n):
    if stats[i, cv2.CC_STAT_AREA] < 300:
        continue
    h = circular_mean_hue(hsv[..., 0][labels == i])
    name = classify_hue(h)
    counts[name] = counts.get(name, 0) + 1
    cx, cy = centroids[i].astype(int)
    cv2.putText(vis, f"{name} {h:.0f}", (cx - 30, cy + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 2)
    cv2.putText(vis, f"{name} {h:.0f}", (cx - 30, cy + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [1.3, 1]})
axes[0].imshow(vis); axes[0].axis("off"); axes[0].set_title("each blob labelled with its class and circular-mean hue")
bar_colors = ["#d93025", "#fa7b17", "#fbbc04", "#188038", "#1a73e8", "#a142f4", "#9aa0a6"]
axes[1].bar(list(counts), list(counts.values()), color=bar_colors[:len(counts)])
axes[1].set_ylabel("candies"); axes[1].set_title("sorting report")
plt.suptitle("Figure 6.2 - Robust colour sorting: segment objects, then classify each blob", fontweight="bold")
plt.tight_layout(); plt.show()
print("Dark brown candies have V < MIN_V and are deliberately ignored: hue is unreliable for dark pixels.")

### Interactive: tune an HSV range yourself
This is exactly how colour inspection systems are commissioned on a production line. Try isolating only the **green** candies, then only the **red** ones (tick *wrap* for red).

In [ ]:
def hsv_demo(h_low=36, h_high=85, min_s=90, min_v=100, wrap=False):
    ranges = [(h_low, 179), (0, h_high)] if wrap else [(h_low, h_high)]
    m = colour_mask(hsv, ranges, min_s, min_v)
    show(m, cv2.bitwise_and(smarties, smarties, mask=m), titles=[f"mask H {h_low}-{h_high}{' (wrapped)' if wrap else ''}",
                                                               "selected pixels"], figsize=(12, 4.6))


interact(hsv_demo, h_low=widgets.IntSlider(value=36, min=0, max=179), h_high=widgets.IntSlider(value=85, min=0, max=179),
         min_s=widgets.IntSlider(value=90, min=0, max=255, step=5), min_v=widgets.IntSlider(value=100, min=0, max=255, step=5),
         wrap=False)

## 7. Histograms

A **histogram** counts how many pixels fall in each intensity bin: $h[k] = \#\{(r, c) : I[r, c] = k\}$. Normalised by the number
of pixels it is a probability distribution; its running sum is the **cumulative distribution function (CDF)**. Histograms reveal
under/over-exposure, contrast and the number of distinct "populations" in an image, and they drive thresholding (Week 4) and
equalisation (Week 3).

In [ ]:
gray = cv2.cvtColor(fruits, cv2.COLOR_BGR2GRAY)
fig, axes = plt.subplots(1, 4, figsize=(21, 4))
axes[0].imshow(gray, cmap="gray"); axes[0].axis("off"); axes[0].set_title("greyscale")
hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).ravel()
axes[1].bar(np.arange(256), hist, width=1, color="#5f6368")
ax2 = axes[1].twinx(); ax2.plot(np.cumsum(hist) / hist.sum(), color="#d93025", lw=2); ax2.set_ylabel("CDF", color="#d93025")
axes[1].set_title("grey-level histogram and CDF")
for i, col in enumerate(["#1a73e8", "#188038", "#d93025"]):
    axes[2].plot(cv2.calcHist([fruits], [i], None, [256], [0, 256]), color=col, label="BGR"[i])
axes[2].set_title("per-channel histograms"); axes[2].legend()
hsv_f = cv2.cvtColor(fruits, cv2.COLOR_BGR2HSV)
h2d = cv2.calcHist([hsv_f], [0, 1], None, [90, 64], [0, 180, 0, 256])
axes[3].imshow(np.log1p(h2d), aspect="auto", origin="lower", extent=[0, 256, 0, 180], cmap="magma")
axes[3].set_xlabel("saturation"); axes[3].set_ylabel("hue"); axes[3].set_title("2D hue-saturation histogram (log)")
plt.suptitle("Figure 7.1 - Histograms of fruits.jpg", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
dark = cv2.convertScaleAbs(gray, alpha=0.4, beta=0)
bright = cv2.convertScaleAbs(gray, alpha=0.5, beta=150)
low_contrast = cv2.convertScaleAbs(gray, alpha=0.3, beta=90)
fig, axes = plt.subplots(2, 4, figsize=(20, 6.5), gridspec_kw={"height_ratios": [2.2, 1]})
for j, (im, name) in enumerate([(gray, "normal"), (dark, "under-exposed"), (bright, "over-exposed"), (low_contrast, "low contrast")]):
    axes[0, j].imshow(im, cmap="gray", vmin=0, vmax=255); axes[0, j].set_title(name); axes[0, j].axis("off")
    axes[1, j].hist(im.ravel(), bins=64, range=(0, 256), color="#1a73e8"); axes[1, j].set_yticks([])
plt.suptitle("Figure 7.2 - Reading exposure and contrast from the histogram", fontweight="bold")
plt.tight_layout(); plt.show()

## 8. Comparing histograms and back-projection

A colour histogram is a compact **descriptor** of an object's colour, independent of its position and shape. Two histograms can be
compared, for example with the **correlation** measure used by `cv2.compareHist` (1 = identical, 0 = unrelated).

**Back-projection** runs this the other way round: given a colour model (a histogram of a sample region), it replaces every pixel by
how typical its colour is of the model. The result is a probability-like map that highlights where the object is. This was the
basis of the classic CamShift tracker (Week 10).

In [ ]:
def hue_hist(img_bgr, bins=36):
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv_img, np.array([0, 60, 40]), np.array([179, 255, 255]))   # ignore grey/dark pixels
    h = cv2.calcHist([hsv_img], [0], mask, [bins], [0, 180])
    return cv2.normalize(h, h, 1, 0, cv2.NORM_L1)


samples = {"apple": apple, "orange": orange, "fruits": fruits, "smarties": smarties}
hists = {k: hue_hist(v) for k, v in samples.items()}
fig, axes = plt.subplots(1, 2, figsize=(16, 4.4))
centres = np.arange(36) * 5 + 2.5
for (name, h), color in zip(hists.items(), ["#d93025", "#fa7b17", "#188038", "#1a73e8"]):
    axes[0].plot(centres, h.ravel(), "o-", ms=3, color=color, label=name)
axes[0].set_xlabel("hue (OpenCV 0-179)"); axes[0].set_ylabel("fraction of coloured pixels"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title("hue histograms")
names = list(hists)
M = np.array([[cv2.compareHist(hists[a], hists[b], cv2.HISTCMP_CORREL) for b in names] for a in names])
im = axes[1].imshow(M, cmap="viridis", vmin=-0.2, vmax=1)
axes[1].set_xticks(range(4), names); axes[1].set_yticks(range(4), names)
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", color="w" if M[i, j] < 0.6 else "k")
plt.colorbar(im, ax=axes[1]); axes[1].set_title("histogram correlation (cv2.compareHist)")
plt.suptitle("Figure 8.1 - Colour histograms as descriptors", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
model_hsv = cv2.cvtColor(orange[180:330, 200:350], cv2.COLOR_BGR2HSV)            # a patch of orange peel
model = cv2.calcHist([model_hsv], [0, 1], None, [45, 32], [0, 180, 0, 256])
cv2.normalize(model, model, 0, 255, cv2.NORM_MINMAX)
backproj = cv2.calcBackProject([hsv_f], [0, 1], model, [0, 180, 0, 256], 1)
backproj = cv2.filter2D(backproj, -1, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
_, found = cv2.threshold(backproj, 50, 255, cv2.THRESH_BINARY)
show(orange[180:330, 200:350], fruits, backproj, cv2.bitwise_and(fruits, fruits, mask=found),
     titles=["colour model: orange peel", "search image", "back-projection (how orange-like)", "thresholded result"],
     figsize=(21, 4.4))

## 9. White balance

A warm tungsten lamp makes everything look yellow-orange; a cool LED makes it blue. Cameras compensate with **white balance**:
scale the channels so that a neutral surface comes out grey. The **grey-world** assumption says the average colour of a
typical scene is grey, so each channel is scaled to the overall mean:

$$ I'_k = I_k \cdot \frac{\bar{I}}{\bar{I}_k}, \qquad \bar{I} = \tfrac13(\bar{I}_B + \bar{I}_G + \bar{I}_R) $$

In machine vision, a better practice is to image a **white or grey reference target** once, under the production lighting, and
store those gains.

In [ ]:
def add_cast(img_bgr, warmth):
    """Simulate a warm (warmth > 0) or cool (warmth < 0) light source."""
    gains = np.array([1 - 0.5 * warmth, 1.0, 1 + 0.5 * warmth]) if warmth >= 0 else np.array([1 - 0.6 * warmth, 1.0, 1 + 0.4 * warmth])
    return np.clip(img_bgr.astype(np.float32) * gains, 0, 255).astype(np.uint8)


def grey_world_demo(img_bgr):
    f = img_bgr.astype(np.float32)
    means = f.reshape(-1, 3).mean(0)
    return np.clip(f * (means.mean() / means), 0, 255).astype(np.uint8)


warm, cool = add_cast(fruits, 0.6), add_cast(fruits, -0.6)
show(warm, grey_world_demo(warm), cool, grey_world_demo(cool),
     titles=["warm (tungsten) cast", "grey-world corrected", "cool (LED) cast", "grey-world corrected"], figsize=(21, 4.4))

### Interactive: colour temperature

In [ ]:
def wb_demo(warmth=0.5, correct=False):
    cast = add_cast(fruits, warmth)
    out = grey_world_demo(cast) if correct else cast
    means = out.reshape(-1, 3).mean(0)
    show(out, titles=[f"channel means B={means[0]:.0f} G={means[1]:.0f} R={means[2]:.0f}"], figsize=(7, 5))


interact(wb_demo, warmth=widgets.FloatSlider(value=0.5, min=-0.9, max=0.9, step=0.05), correct=False)

## 10. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check cell below it.

### E1. Hue range with wrap-around
Return a mask (uint8, 0 or 255) of pixels whose hue lies in `[h_low, h_high]` **inclusive**, with S ≥ `min_s` and V ≥ `min_v`.
If `h_low > h_high`, the range **wraps around** 179 → 0 (e.g. `h_low=170, h_high=10` selects red).

In [ ]:
def hue_mask(hsv_img, h_low, h_high, min_s=50, min_v=50):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    grid = np.zeros((3, 180, 3), np.uint8)
    grid[..., 0] = np.arange(180)
    grid[..., 1] = 200; grid[..., 2] = 200
    grid[1, :, 1] = 10                                   # row 1: unsaturated (grey) pixels
    grid[2, :, 2] = 10                                   # row 2: very dark pixels
    for lo, hi in [(36, 85), (170, 10), (0, 179), (179, 0)]:
        m = hue_mask(grid, lo, hi)
        assert m.dtype == np.uint8 and m.shape == grid.shape[:2], "return a uint8 mask with the image's height and width"
        assert set(np.unique(m)) <= {0, 255}, "mask values must be 0 or 255"
        hues = np.where(m[0] == 255)[0]
        want = np.arange(lo, hi + 1) if lo <= hi else np.r_[np.arange(lo, 180), np.arange(0, hi + 1)]
        assert np.array_equal(np.sort(hues), np.sort(want)), f"range ({lo}, {hi}): wrong set of hues selected"
        assert m[1].max() == 0 and m[2].max() == 0, "grey (low S) and dark (low V) pixels must be excluded"


check("E1 hue_mask", _test_e1)

### E2. Grey-world white balance
Implement grey-world white balance **yourself** (do not call `grey_world_demo`). Return `uint8`, same shape.

In [ ]:
def grey_world(img_bgr):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    for w in (0.7, -0.7):
        out = grey_world(add_cast(fruits, w))
        assert out.dtype == np.uint8 and out.shape == fruits.shape, "return a uint8 image of the same shape"
        m = out.reshape(-1, 3).astype(float).mean(0)
        assert m.max() - m.min() < 4, f"channel means should be (almost) equal after correction, got {np.round(m, 1)}"


check("E2 grey_world", _test_e2)

### E3. Colour similarity score
Return the correlation (`cv2.HISTCMP_CORREL`) between the **normalised 2D hue–saturation histograms** of two BGR images (30 hue bins × 32 saturation bins).

In [ ]:
def colour_similarity(img_a, img_b):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    same = colour_similarity(apple, apple)
    assert abs(same - 1) < 1e-6, f"an image compared with itself should give 1.0, got {same:.3f}"
    flipped = colour_similarity(apple, cv2.flip(apple, 1))
    assert abs(flipped - 1) < 1e-6, "flipping does not change a histogram, so the score should stay 1.0"
    ab, ba = colour_similarity(apple, orange), colour_similarity(orange, apple)
    assert abs(ab - ba) < 1e-6, "the score should be symmetric"
    assert ab < 0.95, "apple vs orange should be clearly less similar than apple vs apple"


check("E3 colour_similarity", _test_e3)

### E4. Robustness experiment (no self-check)
Darken `smarties` to 50% brightness and re-run the colour counter of Section 6. Which colour classes change their count? Fix it by
changing only `MIN_S` / `MIN_V`, and explain why brightness changes affect some colours more than others.

In [ ]:
# E4: your code here

## 11. Challenge: a colour-sorting station

Build a webcam colour sorter (locally, set `RUN_WEBCAM = True`):
1. Hold three differently coloured objects (e.g. Lego bricks, bottle caps) in front of the camera one at a time and record their
   typical H, S, V (use the centre pixel or a small ROI).
2. Build one `inRange` mask per class, count the blobs in each live frame and print a report such as `red: 2, blue: 1`.
3. Test under two lighting conditions (desk lamp on/off). Which colour is least reliable? Would Lab distance to a reference
   colour (ΔE) work better than fixed HSV ranges?

In [ ]:
RUN_WEBCAM = False

if RUN_WEBCAM:
    cap = cv2.VideoCapture(0)
    ok_cam, frame = cap.read()
    cap.release()
    if ok_cam:
        h, w = frame.shape[:2]
        centre = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)[h // 2 - 5:h // 2 + 5, w // 2 - 5:w // 2 + 5].reshape(-1, 3)
        print("centre H, S, V (median):", np.median(centre, axis=0))
        show(frame, titles=["webcam frame"])

## 12. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'Why is HSV usually better than BGR for colour thresholding?',
        "options": [
            'It has more channels',
            'Hue separates colour from brightness, so shading changes mostly V',
            'It is faster to compute',
            'It uses 16 bits per channel',
        ],
        "answer": 'd8d056b4e7',
        "explain": 'A shadow scales R, G and B together, which leaves hue nearly unchanged.',
    },
    {
        "q": 'In OpenCV, what is the range of the H channel for 8-bit images?',
        "options": [
            '0-255',
            '0-360',
            '0-179',
            '0-100',
        ],
        "answer": '9e7b8543a6',
        "explain": 'Degrees are divided by 2 so that 360 degrees fit into a uint8.',
    },
    {
        "q": 'Why does detecting red need two inRange calls in HSV?',
        "options": [
            'Red has two wavelengths',
            'Red hue wraps around: it lies near both 0 and 179',
            'Red is too dark',
            'OpenCV stores red twice',
        ],
        "answer": '75db36324d',
        "explain": 'Combine inRange(0..10) and inRange(170..179) with bitwise OR.',
    },
    {
        "q": 'A Bayer colour camera has 4 million pixels. How many red-filtered pixels does it have?',
        "options": [
            '4 million',
            '2 million',
            '1 million',
            '1.33 million',
        ],
        "answer": '71e1ee868c',
        "explain": 'The Bayer pattern is 50% green, 25% red and 25% blue; the missing values are interpolated (demosaicing).',
    },
    {
        "q": 'You need to check that a painted part matches the reference colour within a perceptual tolerance. Which space?',
        "options": [
            'BGR',
            'HSV',
            'Lab (Delta E)',
            'greyscale',
        ],
        "answer": '366fb2e785',
        "explain": 'Lab is approximately perceptually uniform, so Euclidean distance (Delta E) matches perceived difference.',
    },
    {
        "q": 'A histogram of an image is squeezed into a narrow band around 100-140. The image is...',
        "options": [
            'over-exposed',
            'under-exposed',
            'low contrast',
            'saturated',
        ],
        "answer": '4587342e2b',
        "explain": 'Few distinct grey levels are used. Contrast stretching or equalisation (Week 3) spreads them out.',
    },
    {
        "q": 'What does histogram back-projection produce?',
        "options": [
            'The histogram of the back of the object',
            "A map of how well each pixel's colour matches a colour model",
            'A binary edge map',
            'A 3D reconstruction',
        ],
        "answer": '7fa050da66',
        "explain": "Each pixel is replaced by the model histogram's value for its colour.",
    },
], title='Week 01B quiz')

## 13. Exit ticket (reflection)
1. Give one inspection task where colour is the *right* cue and one where you would deliberately use a monochrome camera (or NIR light).
2. Your colour sorter works in the morning but fails in the afternoon sun from a window. What changed physically, and name two fixes.
3. Why is hue unreliable for white, grey and black objects?

## 14. What's new (2025–26)

> - **Multispectral and hyperspectral cameras** (dozens to hundreds of narrow bands) are becoming affordable for food sorting, plastic
>   recycling and agriculture; they separate materials that look identical in RGB.
> - **RGB-NIR and polarisation sensors** add extra channels on one chip (e.g. to see through glare or detect stress in glass).
> - Learned colour constancy and auto white balance run in real time on phone and edge-camera ISPs, but industrial systems still
>   rely on **controlled lighting and reference targets** for repeatability.

## 15. References and further exploration

**Textbooks and papers**
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §2.3.2 (colour). [szeliski.org/Book](https://szeliski.org/Book/)
- Gonzalez & Woods (2018). *Digital Image Processing*, 4th ed., ch. 6 (colour image processing) and §3.3 (histograms).
- Bayer, B. E. (1976). *Color imaging array.* US Patent 3,971,065. [patents.google.com](https://patents.google.com/patent/US3971065A)
- Buchsbaum, G. (1980). *A spatial processor model for object colour perception* (grey-world). J. Franklin Institute 310(1). [doi:10.1016/0016-0032(80)90058-7](https://doi.org/10.1016/0016-0032(80)90058-7)
- Swain, M. J. & Ballard, D. H. (1991). *Color indexing* (histogram matching and back-projection). IJCV 7. [doi:10.1007/BF00130487](https://doi.org/10.1007/BF00130487)
- Sharma, G., Wu, W. & Dalal, E. N. (2005). *The CIEDE2000 colour-difference formula.* Color Res. Appl. 30(1). [doi:10.1002/col.20070](https://doi.org/10.1002/col.20070)

**Documentation and interactive**
- OpenCV: *Changing colour spaces* and *Histograms*. [docs.opencv.org](https://docs.opencv.org/4.x/df/d9d/tutorial_py_colorspaces.html)
- scikit-image `color` module. [scikit-image.org](https://scikit-image.org/docs/stable/api/skimage.color.html)
- Nayar, S. *First Principles of Computer Vision*: "Image Sensing" lecture (colour sensing, Bayer). [fpcv.cs.columbia.edu](https://fpcv.cs.columbia.edu/)
- Poynton, C. *Colour FAQ*. [poynton.ca](https://poynton.ca/ColorFAQ.html)

**Images:** `smarties.png`, `fruits.jpg`, `apple.jpg`, `orange.jpg` from the OpenCV sample data.

## 16. Key takeaways
- Pixel colour = illuminant × surface × sensor. **Control the light** or compensate with white balance.
- Colour cameras use a **Bayer mosaic** and demosaicing; monochrome cameras give sharper measurements.
- **HSV** separates hue from brightness (good for thresholding; mind the red wrap-around and low-S/low-V pixels);
  **Lab** is perceptually uniform (ΔE tolerances); **YCrCb** separates luma from chroma.
- `inRange` → mask → opening → connected components is a complete colour-sorting pipeline.
- **Histograms** summarise exposure and contrast, act as colour descriptors, and power back-projection.

**Next week (02A):** image formation: the pinhole camera, lenses, depth of field, distortion and **lighting for machine vision**.